# Predict / evaluate / compare loop

The core inference-iteration workflow: hold the model and dataset fixed, vary the **prompt**, and let the numbers pick the winner. An experiment is fully described by **model version + dataset version + prompt version**, so comparing two prompt versions is an apples-to-apples A/B test.

In [ ]:
import localml as ml

ml.configure(api_url="http://localhost:8000", token="local-dev-token")

## 1. Fix the model and dataset

Resolve an existing model version by `name:version` (or register one), and register a JSONL eval set. The `expected` column is what we'll score against.

In [ ]:
version = ml.register_model(
    name="assistant", artifact_uri="models/assistant", framework="mlx"
)
rows = [
    {"question": "2+2?", "expected": "4"},
    {"question": "Capital of France?", "expected": "Paris"},
    {"question": "Color of the sky?", "expected": "blue"},
]
dataset = ml.datasets.register(
    project="local", name="qa-eval", artifact_uri="datasets/qa.jsonl", rows=rows
)
dataset.name, dataset.version, dataset.row_count

## 2. Register two prompt variants

A terse prompt versus a chain-of-thought prompt. Registering the same name twice auto-increments the version (`qa:v1`, `qa:v2`).

In [ ]:
terse = ml.prompts.register(name="qa", template="Q: {question}\nA:")
cot = ml.prompts.register(
    name="qa", template="Q: {question}\nThink step by step, then answer.\nA:"
)
terse.version, cot.version

## 3. Predict with both prompts

Two prediction jobs, identical except for the prompt version. Inference runs once per job and outputs are stored as JSONL, so we can (re-)score without re-inferring.

In [ ]:
pred_a = ml.predict(model=version, dataset=dataset, prompt=terse, provider="echo").wait()
pred_b = ml.predict(model=version, dataset=dataset, prompt=cot, provider="echo").wait()
pred_a.status, pred_b.status

## 4. Score each variant

Same metrics, same `expected_field`, so the two evaluations are directly comparable.

In [ ]:
metrics = ["exact_match", "latency_p95"]
cfg = {"expected_field": "expected"}
eval_a = ml.evals.run(pred_a, metrics, config=cfg).wait()
eval_b = ml.evals.run(pred_b, metrics, config=cfg).wait()
eval_a.metrics, eval_b.metrics

## 5. Compare and pick the winner

`ml.compare` aligns the two jobs on `example_id`: what varied (here, the prompt version), output agreement, latency delta, and per-metric a/b/delta values.

In [ ]:
report = ml.compare(eval_a, eval_b)
print("differs:", report.differs)
print("metrics:", report.metrics)

winner = (
    "cot" if (eval_b.metrics or {}).get("exact_match", 0)
    > (eval_a.metrics or {}).get("exact_match", 0) else "terse"
)
print("winner:", winner)

## Iterate

Register the next prompt variant, predict, evaluate, and compare against the current champion. Because prediction and evaluation are decoupled, you can also add a new metric and re-score existing prediction jobs without re-running inference.